In [5]:
import os
import json
import yaml
import queue

In [13]:
class Keyword:
    def __init__(self, keyObj):
        self.name = keyObj['m_Name']
        self.defaultRefName = keyObj['m_DefaultReferenceName']
        self.overrideRefName = keyObj['m_OverrideReferenceName']
    def getKeyword(self):
        if self.overrideRefName != '':
            return self.overrideRefName[:-3] # remove the _ON postfix
        else:
            return self.defaultRefName[:-3] # remove the _ON postfix

class Property:
    def __init__(self, propObj, propType):
        if propType == 'UnityEditor.ShaderGraph.Internal.ColorShaderProperty':
            self.type = 'float4'
        elif propType == 'UnityEditor.ShaderGraph.Internal.Texture2DShaderProperty':
            self.type = 'TextureInfo'
        elif propType == 'UnityEditor.ShaderGraph.Internal.BooleanShaderProperty':
            self.type = 'bool'
        elif propType == 'UnityEditor.ShaderGraph.Internal.Vector1ShaderProperty':
            self.type = 'float'
        elif propType == 'UnityEditor.ShaderGraph.Internal.Vector2ShaderProperty':
            self.type = 'float2'
        elif propType == 'UnityEditor.ShaderGraph.Internal.Vector3ShaderProperty':
            self.type = 'float3'
        elif propType == 'UnityEditor.ShaderGraph.Internal.Vector4ShaderProperty':
            self.type = 'float4'
        else: 
            print('error: Property type %s not implemented' % propType)
        self.name = propObj['m_Name']
        self.defaultRefName = propObj['m_DefaultReferenceName']
        self.overrideRefName = propObj['m_OverrideReferenceName']

    def getName(self):
        if self.overrideRefName != '':
            return self.overrideRefName
        else:
            return self.defaultRefName

class Slot:
    def __init__(self, jsonDoc, slotType):
        self.id = jsonDoc['m_Id']
        self.slotType = jsonDoc['m_SlotType']
        self.stage = jsonDoc['m_StageCapability']
        self.name = jsonDoc['m_ShaderOutputName']
        self.sgType = slotType
        if slotType in slotTypes:
            self.hlslType = slotTypes[slotType]
        else:
            #print('warning: slot type %s not implemented!' % self.sgType)
            self.hlslType = 'None'
        self.values = []
        if 'm_Value' in jsonDoc:
            if isinstance(jsonDoc['m_Value'], dict):
                for v in jsonDoc['m_Value'].values():
                    self.values.append(v)
            else:
                self.values.append(jsonDoc['m_Value'])
        self.visited = True
    def getCode(self, outType):
        if self.sgType == 'UnityEditor.ShaderGraph.ColorRGBMaterialSlot':
            return 'float3(' + str(self.values[0]) + ', ' + str(self.values[1]) + ', ' + str(self.values[2]) + ')'
        elif self.sgType == 'UnityEditor.ShaderGraph.UVMaterialSlot':
            return 'i.uv_texcoord'
        elif self.sgType == 'UnityEditor.ShaderGraph.DynamicVectorMaterialSlot' or self.sgType == 'UnityEditor.ShaderGraph.DynamicValueMaterialSlot':
            if outType == 'float':
                return str(self.values[0])
            elif outType == 'float2':
                return 'float2(' + str(self.values[0]) + ', ' + str(self.values[1]) + ')'
            elif outType == 'float3':
                return 'float3(' + str(self.values[0]) + ', ' + str(self.values[1]) + ', ' + str(self.values[2]) + ')'
            elif outType == 'float4':
                return 'float4(' + str(self.values[0]) + ', ' + str(self.values[1]) + ', ' + str(self.values[2]) + ', ' + str(self.values[3]) + ')'
        elif self.sgType == 'UnityEditor.ShaderGraph.Vector1MaterialSlot':
            return str(self.values[0])
        elif self.sgType == 'UnityEditor.ShaderGraph.Vector2MaterialSlot':
            return 'float2(' + str(self.values[0]) + ', ' + str(self.values[1]) + ')'
        elif self.sgType == 'UnityEditor.ShaderGraph.Vector3MaterialSlot':
            return 'float3(' + str(self.values[0]) + ', ' + str(self.values[1]) + ', ' + str(self.values[2]) + ')'
        elif self.sgType == 'UnityEditor.ShaderGraph.SamplerStateMaterialSlot':
            return ''
        return self.sgType + ' not implemented!'

codeTemplates = {
    'UnityEditor.ShaderGraph.PropertyNode' : '',
    'UnityEditor.ShaderGraph.KeywordNode' : '%s ? %s : %s',
    'UnityEditor.ShaderGraph.SampleTexture2DNode' : 'tex2D(%s, %s)',
    'UnityEditor.ShaderGraph.LerpNode' : 'lerp(%s, %s, %s)',
    'UnityEditor.ShaderGraph.Vector1Node' : '%s',
    'UnityEditor.ShaderGraph.Vector2Node' : 'float2(%s, %s)',
    'UnityEditor.ShaderGraph.Vector3Node' : 'float3(%s, %s, %s)',
    'UnityEditor.ShaderGraph.Vector4Node' : 'float4(%s, %s, %s, %s)',
    'UnityEditor.ShaderGraph.MultiplyNode' : '%s * %s',
    'UnityEditor.ShaderGraph.SplitNode' : '',
    'UnityEditor.ShaderGraph.BlockNode' : '%s',
    'UnityEditor.ShaderGraph.TilingAndOffsetNode' : '%s * %s + %s',
    'UnityEditor.ShaderGraph.NormalStrengthNode' : 'ScaleNormal(%s, %s)',
    'UnityEditor.ShaderGraph.NormalBlendNode' : 'BlendNormals(%s, %s)',
    'UnityEditor.ShaderGraph.SubGraphNode' : '',
    'UnityEditor.ShaderGraph.SubGraphOutputNode' : '',
    'UnityEditor.ShaderGraph.PBRMasterNode' : '',
    'UnityEditor.ShaderGraph.BranchNode' : '%s ? %s : %s',
    'UnityEditor.ShaderGraph.IsFrontFaceNode' : 'i.isFrontFacing',
    'UnityEditor.ShaderGraph.NegateNode' : '-%s',
    'UnityEditor.ShaderGraph.ViewDirectionNode' : 'normalize(cameraPosition.xyz - i.worldPos)',
    'UnityEditor.ShaderGraph.DotProductNode' : 'dot(%s, %s)',
    'UnityEditor.ShaderGraph.PowerNode' : 'pow(%s, %s)',
    'UnityEditor.ShaderGraph.RemapNode' : 'remap(%s, %s, %s)',
    'UnityEditor.ShaderGraph.NormalizeNode' : 'normalize(%s)',
    'UnityEditor.ShaderGraph.MaximumNode' : 'max(%s, %s)',
    'UnityEditor.ShaderGraph.SineNode' : 'sin(%s)',
    'UnityEditor.ShaderGraph.CosineNode' : 'cos(%s)',
    'UnityEditor.ShaderGraph.NoiseNode' : 'float(%s * %s)',
    'UnityEditor.ShaderGraph.PositionNode' : 'i.worldPos',
    'UnityEditor.ShaderGraph.CombineNode' : '',
    'UnityEditor.ShaderGraph.AddNode' : '%s + %s',
    'UnityEditor.ShaderGraph.TimeNode' : '',
    'UnityEditor.ShaderGraph.OneMinusNode' : '1.0 - %s',
    'UnityEditor.ShaderGraph.RedirectNodeData' : '',
    'UnityEditor.ShaderGraph.TransformNode' : '',
    'UnityEditor.ShaderGraph.AbsoluteNode' : 'abs(%s)'
}

slotTypes = {
    'UnityEditor.ShaderGraph.Texture2DInputMaterialSlot' : 'TextureInfo',
    'UnityEditor.ShaderGraph.Texture2DMaterialSlot' : 'TextureInfo',
    'UnityEditor.ShaderGraph.UVMaterialSlot' : 'float2',
    'UnityEditor.ShaderGraph.ColorRGBMaterialSlot' : 'float3',
    'UnityEditor.ShaderGraph.NormalMaterialSlot' : 'float3',
    'UnityEditor.ShaderGraph.BooleanMaterialSlot' : 'bool',
    'UnityEditor.ShaderGraph.Vector1MaterialSlot' : 'float',
    'UnityEditor.ShaderGraph.Vector2MaterialSlot' : 'float2',
    'UnityEditor.ShaderGraph.Vector3MaterialSlot' : 'float3',
    'UnityEditor.ShaderGraph.Vector4MaterialSlot' : 'float4',
    'UnityEditor.ShaderGraph.DynamicVectorMaterialSlot' : 'None',
    'UnityEditor.ShaderGraph.DynamicValueMaterialSlot' : 'None'
}

class Node:
    def __init__(self, jsonDoc, nodeID, nodeType):
        self.name = jsonDoc['m_Name']
        self.propName = ''
        self.id = nodeID
        self.type = nodeType
        self.slots = []
        self.vars = {}
        self.finished = False
        self.isOutputNode = False
    
    def addSlot(self, slot):
        self.slots.append(slot)
    def getSlot(self, slotID):
        for slot in self.slots:
            if slot.id == slotID:
                return slot
    def getInputSlots(self):
        inSlots= []
        for slot in self.slots:
            if slot.slotType == 0:
                inSlots.append(slot.id)
        return inSlots
    def getOutputSlots(self):
        outSlots = []
        for slot in self.slots:
            if slot.slotType == 1:
                outSlots.append(slot.id)
        return outSlots
    def visit(self, inName, inType, inSlot):
        self.vars[inSlot] = (inName, inType)
        for slot in self.slots:
            if slot.id == inSlot:
                slot.visited = True
        self.finished = True
        for slot in self.slots:
            if slot.slotType == 0:
                if slot.visited == False:
                    self.finished = False
    def getType(self):
        types = []
        for name, type in self.vars.values():
            if type != 'bool':
                types.append(type)
        outSlots = self.getOutputSlots()
        if len(outSlots) == 1:
            outType = self.getSlot(outSlots[0]).hlslType
            if outType != 'None':
                return outType
        if len(types) == 0:
            return 'float4'

        vecTypes = ['float2', 'float3', 'float4']
        # if all input types are the same we dont need any casts so outType = inType
        sameType = True
        for i in range(len(types)-1):
            if types[i] != types[i+1]:
                sameType = False
                break
        if sameType:
            return types[0]

        # we have different input types, we need to determine outType and add casts if needed        
        lowestVecType = 2
        for name, type in self.vars.values():
            for i,t in enumerate(vecTypes):
                if t == type:
                    lowestVecType = min(lowestVecType, i)
        return vecTypes[lowestVecType]

    def getInputVars(self, outType):
        # check if we need to add a cast
        for slotID, var in self.vars.items():
            slot = self.getSlot(slotID)
            if slotID >= 0 and slot.slotType == 0:
                name, type = var
                if slot.hlslType == 'None':
                    if type != outType and self.slots[self.getOutputSlots()[0]].hlslType == 'None':
                        self.vars[slotID] = ('(' + outType + ')' + name, type)
                    continue
                if type != slot.hlslType:
                    self.vars[slotID] = ('(' + slot.hlslType + ')' + name, type)

        # get input vars from nodes and slots
        inVars = {}
        if -1 in self.vars: # we have a keyword
            inVars[-1] = self.vars[-1][0]
        for slot in self.slots:
            if slot.slotType == 0:
                if slot.id in self.vars:
                    inVars[slot.id] = self.vars[slot.id][0]
                else:
                    slotCode = slot.getCode(outType)
                    if slotCode != '':
                        inVars[slot.id] = slotCode
        return inVars
                        
    def getCode(self, varIdx):
        #print(self.type)
        outType = self.getType()
        inVars = self.getInputVars(outType)
        values = []
        for v in inVars.values():
            values.append(v)
        code = ''
        if self.isOutputNode:
            outName = 'o.' + self.slots[0].name # TODO: add prefix
        else: 
            code += outType + ' '
            outName = 'var' + str(varIdx)
        if codeTemplates[self.type] == '':
            code = ''
            outName, outType = self.vars[self.slots[0].id]
        else:
            code += outName + ' = ' + codeTemplates[self.type] % tuple(values) + ';\n'
        return [code], outName, outType
        
    def getOutputVars(self, outName, outType):
        outVars = {}
        for slot in self.slots:
            if slot.slotType == 1:
                name = outName
                if len(self.getOutputSlots()) > 1 and slot.hlslType != outType:
                    swizzle = slot.name.lower()
                    name += '.' + swizzle
                if slot.hlslType == 'None':
                    outVars[slot.id] = (name, outType)
                else:
                    outVars[slot.id] = (name, slot.hlslType)
        return outVars

class TransformNode(Node):
    def __init__(self, jsonDoc, nodeID, nodeType):
        Node.__init__(self, jsonDoc, nodeID, nodeType)
        self.inSpace = jsonDoc['m_Conversion']['from']
        self.outSpace = jsonDoc['m_Conversion']['to']
        self.convType = jsonDoc['m_ConversionType']
    def getCode(self, varIdx):
        outName = 'var' + str(varIdx)
        outType = 'float3'
        inVars = self.getInputVars(outType)
        name = inVars[0]
        if self.inSpace == 0 and self.outSpace == 2: # object to world
            #code = outType + ' ' + outName + ' = mat3(model.localToWorldNormal) * ' + name  + ';\n'
            code = outType + ' ' + outName + ' = mul(' + name + ', (float3x3)localToWorldNormal);\n'
        elif self.inSpace == 3 and self.outSpace == 0: # tangent to object
            #code = outType + ' ' + outName + ' = inverse(mat3(model.localToWorldNormal)) * (i.wTBN * ' + name  + ');\n'
            code = outType + ' ' + outName + ' = mul(mul((float3)' + name + ', i.wTBN), (float3x3)localToWorld);\n'
        else:
            print('error: TransformNode %d -> %d not implemented' % (self.inSpace, self.outSpace))
        return [code], outName, outType

class CombineNode(Node):
    def __init__(self, jsonDoc, nodeID, nodeType):
        Node.__init__(self, jsonDoc, nodeID, nodeType)
    def getCode(self, varIdx):
        outName = 'var' + str(varIdx)
        outType = 'float4'
        inVars = self.getInputVars(outType)
        values = []
        for v in inVars.values():
            values.append(v)
        code = outType + ' ' + outName + ' = '
        code += codeTemplates['UnityEditor.ShaderGraph.Vector4Node'] % tuple(values) + ';\n'      
        return [code], outName, outType

class SampleTexture2DNode(Node):
    def __init__(self, jsonDoc, nodeID, nodeType):
        Node.__init__(self, jsonDoc, nodeID, nodeType)
        self.textureType = jsonDoc['m_TextureType']
    def getCode(self, varIdx):
        outName = 'var' + str(varIdx)
        outType = 'float4'
        inVars = self.getInputVars(outType)
        values = []
        for v in inVars.values():
            values.append(v)
        code = outType + ' ' + outName + ' = '
        if self.textureType == 0:
            code += codeTemplates[self.type] % tuple(values) + ';\n'
        elif self.textureType == 1:
            code += 'UnpackNormal4(%s, %s)' % tuple(values) + ';\n'
        return [code], outName, outType

class PBRMasterNode(Node):
    def __init__(self, jsonDoc, nodeID, nodeType):
        Node.__init__(self, jsonDoc, nodeID, nodeType)
    def getCode(self, varIdx):
        outName = 'var' + str(varIdx)
        outType = 'None'
        inVars = self.getInputVars(outType)
        code = []
        for slot in self.slots:
            if slot.slotType == 0 and slot.stage == 2:
                name = inVars[slot.id]
                code.append('o.' + slot.name + ' = ' + name + ';\n')
        return code, outName, outType

class SubGraphNode(Node):
    def __init__(self, jsonDoc, nodeID, nodeType):
        Node.__init__(self, jsonDoc, nodeID, nodeType)
        subGraphJson = json.loads(jsonDoc['m_SerializedSubGraph'])
        self.guid = subGraphJson['subGraph']['guid']
    def getCode(self, varIdx):
        funcName = self.name.replace('-', '')
        inStructName = funcName + 'DataIn'
        outStructName = funcName + 'DataOut'
        inStructVar = inStructName[0].lower() + inStructName[1:] + str(varIdx)
        outStructVar = outStructName[0].lower() + outStructName[1:] + str(varIdx)
        code = []
        code.append(inStructName + ' ' + inStructVar + ';\n')
        code.append(outStructName + ' ' + outStructVar + ';\n')
        for slot in self.slots:
            if slot.slotType == 0:
                name, type = self.vars[slot.id]
                code.append(inStructVar + '.' + slot.name + ' = ' + name + ';\n')
        code.append(funcName + '(i, ' + inStructVar + ', ' + outStructVar + ');\n')
        outName = outStructVar
        outType = outStructName
        return code, outName, outType
        
    def getOutputVars(self, outName, outType):
        outVars = {}
        for slot in self.slots:
            if slot.slotType == 1:
                name = outName
                if slot.hlslType != outType:
                    swizzle = slot.name
                    if len(swizzle) <= 4:
                        swizzle = swizzle.lower()
                    name += '.' + swizzle
                if slot.hlslType == 'None':
                    outVars[slot.id] = (name, outType)
                else:
                    outVars[slot.id] = (name, slot.hlslType)
        return outVars
        
class SubGraphOutputNode(Node):
    def __init__(self, jsonDoc, nodeID, nodeType):
        Node.__init__(self, jsonDoc, nodeID, nodeType)
    def getCode(self, varIdx):
        outName = 'var' + str(varIdx)
        outType = 'None'
        code = []
        for slot in self.slots:
            if slot.slotType == 0:
                name, type = self.vars[slot.id]
                slotCode = 'outData.' + slot.name + ' = '
                if type != slot.hlslType:
                    slotCode += slot.hlslType + '(' + name + ')'
                else:
                    slotCode += name
                slotCode += ';\n'
                code.append(slotCode)
        return code, outName, outType

class TimeNode(Node):
    def __init__(self, jsonDoc, nodeID, nodeType):
        Node.__init__(self, jsonDoc, nodeID, nodeType)
    def getCode(self, varIdx):
        outName = 'var' + str(varIdx)
        outType = 'float'
        code = outType + ' ' + outName + ' = time.y;\n'
        return [code], outName, outType

class OutputNode:
    def __init__(self, name, type, slotID, objectID):
        self.name = name
        self.type = type
        self.slotID = slotID
        self.objectID = objectID

In [15]:
class ShaderGraph:
    def __init__(self, filepath, inPrefix = '', outPrefix = ''):

        self.name = filepath.split('/')[-1].split('.')[0]
        self.inPrefix = inPrefix
        self.outPrefix = outPrefix

        self.nodes = {}
        self.edgeMapIn = {}
        self.edgeMapOut = {}
        self.properties = {}
        self.keywords = {}
        self.outputNodes = []

        with open(filepath, 'r') as f:
            lines = f.readlines()

        if lines[1].find('m_SerializedProperties') > -1:
            self.loadFromJSONSerialized(filepath)
        else:
            self.loadFromJSON(filepath)

    def loadFromJSON(self, filepath):
        with open(filepath, 'r') as f:
            lines = f.readlines()
            
        jsonDocs = []
        content = ''
        for l in lines:
            content += l
            if l == '}\n':
                jsonDocs.append(content)
                content = ''
        
        header = json.loads(jsonDocs[0])

        objects = {}
        for doc in jsonDocs[1:]:
            content = json.loads(doc)
            objectID = content['m_ObjectId']
            objects[objectID] = content
        
        for prop in header['m_Properties']:
            propID = prop['m_Id']
            propObj = objects[propID]
            propType = propObj['m_Type']
            self.properties[propID] = Property(propObj, propType)

        for key in header['m_Keywords']:
            keyID = key['m_Id']
            keyObj = objects[keyID]
            self.keywords[keyID] = Keyword(keyObj)
        
        self.nodes = {}
        for n in header['m_Nodes']:
            nodeID = n['m_Id']
            nodeObj = objects[nodeID]
            nodeType = nodeObj['m_Type']
            if nodeType == 'UnityEditor.ShaderGraph.TimeNode':
                 node = TimeNode(nodeObj, nodeID, nodeType)
            elif nodeType == 'UnityEditor.ShaderGraph.TransformNode':
                node = TransformNode(nodeObj, nodeID, nodeType)
            elif nodeType == 'UnityEditor.ShaderGraph.CombineNode':
                node = CombineNode(nodeObj, nodeID, nodeType)
            elif nodeType == 'UnityEditor.ShaderGraph.SampleTexture2DNode':
                node = SampleTexture2DNode(nodeObj, nodeID, nodeType)
            elif nodeType == 'UnityEditor.ShaderGraph.PBRMasterNode':
                node = PBRMasterNode(nodeObj, nodeID, nodeType)
            elif nodeType == 'UnityEditor.ShaderGraph.SubGraphNode':
                node = SubGraphNode(nodeObj, nodeID, nodeType)
            elif nodeType == 'UnityEditor.ShaderGraph.SubGraphOutputNode':
                node = SubGraphOutputNode(nodeObj, nodeID, nodeType)
            else:
                node = Node(nodeObj, nodeID, nodeType)

            for slotEntry in nodeObj['m_Slots']:
                slotID = slotEntry['m_Id']
                slotObj = objects[slotID]
                slotType = slotObj['m_Type']
                slot = Slot(slotObj, slotType)
                node.addSlot(slot)

            if nodeType == 'UnityEditor.ShaderGraph.BlockNode':
                node.isOutputNode = True
                # TODO: create a different unity surface depending on the out node type
                if node.slots[0].name == 'BaseColor':
                    node.slots[0].name = 'Albedo'
                if node.slots[0].name == 'NormalTS' or node.slots[0].name == 'NormalOS':
                    node.slots[0].name = 'Normal'

            if nodeType == 'UnityEditor.ShaderGraph.PropertyNode':
                propID = nodeObj['m_Property']['m_Id']
                prop = self.properties[propID]
                node.propName = self.inPrefix + prop.getName()
            if nodeType == 'UnityEditor.ShaderGraph.KeywordNode':
                keyID = nodeObj['m_Keyword']['m_Id']
                keyword = self.keywords[keyID]
                keyName = keyword.getKeyword()
                node.visit(keyName, 'bool', -1)
            self.nodes[nodeID] = node

            if nodeType not in codeTemplates:
                print('error node type %s not implemented!' % nodeType)

        for e in header['m_Edges']:
            inputSlot = e['m_InputSlot']
            outputSlot = e['m_OutputSlot']
            input = (inputSlot['m_Node']['m_Id'], inputSlot['m_SlotId'])
            output = (outputSlot['m_Node']['m_Id'], outputSlot['m_SlotId'])
            if output not in self.edgeMapOut:
                self.edgeMapOut[output] = []
            self.edgeMapOut[output].append(input)
            self.edgeMapIn[input] = output

        for guid, node in self.nodes.items():
            for inList in self.edgeMapOut.values():
                for inID, inSlot in inList:
                    if guid == inID:
                        for slot in node.slots:
                            if slot.id == inSlot:
                                slot.visited = False

        outputNode = header['m_OutputNode']['m_Id']
        if outputNode == '':
            for idx, block in enumerate(header['m_FragmentContext']['m_Blocks']):
                blockID = block['m_Id']
                blockNode = self.nodes[blockID]
                slot = blockNode.slots[0]
                slotName = slot.name
                slotType = slot.hlslType
                self.outputNodes.append(OutputNode(slotName, slotType, slot.id, blockID))
        else:
            outNode = self.nodes[outputNode]
            for slot in outNode.slots:
                if slot.stage == 2 or slot.stage == 3:
                    self.outputNodes.append(OutputNode(slot.name, slot.hlslType, slot.id, outputNode))
                                
    def loadFromJSONSerialized(self, filepath):
        with open(filepath, 'r') as f:
            jsonDoc = json.loads(f.read())

        for prop in jsonDoc['m_SerializedProperties']:
            propType = prop['typeInfo']['fullName']
            propObj = json.loads(prop['JSONnodeData'])
            guid = propObj['m_Guid']['m_GuidSerialized']
            self.properties[guid] = Property(propObj, propType)

        for node in jsonDoc['m_SerializableNodes']:
            nodeType = node['typeInfo']['fullName']
            nodeObj = json.loads(node['JSONnodeData'])
            nodeID = nodeObj['m_GuidSerialized']
            if nodeType == 'UnityEditor.ShaderGraph.TimeNode':
                 node = TimeNode(nodeObj, nodeID, nodeType)
            elif nodeType == 'UnityEditor.ShaderGraph.TransformNode':
                node = TransformNode(nodeObj, nodeID, nodeType)
            elif nodeType == 'UnityEditor.ShaderGraph.CombineNode':
                node = CombineNode(nodeObj, nodeID, nodeType)
            elif nodeType == 'UnityEditor.ShaderGraph.SampleTexture2DNode':
                node = SampleTexture2DNode(nodeObj, nodeID, nodeType)
            elif nodeType == 'UnityEditor.ShaderGraph.PBRMasterNode':
                node = PBRMasterNode(nodeObj, nodeID, nodeType)
            elif nodeType == 'UnityEditor.ShaderGraph.SubGraphNode':
                node = SubGraphNode(nodeObj, nodeID, nodeType)
            elif nodeType == 'UnityEditor.ShaderGraph.SubGraphOutputNode':
                node = SubGraphOutputNode(nodeObj, nodeID, nodeType)
            else:
                node = Node(nodeObj, nodeID, nodeType)

            if nodeType not in codeTemplates:
                print('error node type %s not implemented!' % nodeType)

            for slotEntry in nodeObj['m_SerializableSlots']:
                slotObj = json.loads(slotEntry['JSONnodeData'])
                slotType = slotEntry['typeInfo']['fullName']
                slot = Slot(slotObj, slotType)
                node.addSlot(slot)

            if nodeType == 'UnityEditor.ShaderGraph.BlockNode':
                node.isOutputNode = True
            if nodeType == 'UnityEditor.ShaderGraph.PropertyNode':
                if 'm_Property' in nodeObj:
                    propID = nodeObj['m_Property']['m_Id']
                elif 'm_PropertyGuidSerialized' in nodeObj:
                    propID = nodeObj['m_PropertyGuidSerialized']
                prop = self.properties[propID]
                node.propName = self.inPrefix + prop.getName()
                
            self.nodes[nodeID] = node

        for e in jsonDoc['m_SerializableEdges']:
            edgeObj = json.loads(e['JSONnodeData'])
            inputSlot = edgeObj['m_InputSlot']
            outputSlot = edgeObj['m_OutputSlot']
            input = (inputSlot['m_NodeGUIDSerialized'], inputSlot['m_SlotId'])
            output = (outputSlot['m_NodeGUIDSerialized'], outputSlot['m_SlotId'])
            if output not in self.edgeMapOut:
                self.edgeMapOut[output] = []
            self.edgeMapOut[output].append(input)
            self.edgeMapIn[input] = output
            
        for guid, node in self.nodes.items():
            for inList in self.edgeMapOut.values():
                for inID, inSlot in inList:
                    if guid == inID:
                        for slot in node.slots:
                            if slot.id == inSlot:
                                slot.visited = False

       # TODO: can there be a fragment context in serialized json?
        guid = jsonDoc['m_ActiveOutputNodeGuidSerialized']
        if guid =='':
            for guid, node in self.nodes.items():
                if type(node) == SubGraphOutputNode:
                    for slot in node.slots:
                        #print(slot.name, slot.type, slot.id)
                        self.outputNodes.append(OutputNode(slot.name, slot.hlslType, slot.id, guid))
        else:
            outNode = self.nodes[guid]
            for slot in outNode.slots:
                if slot.stage == 2:
                    #print(slot.name, slot.type, slot.id)
                    self.outputNodes.append(OutputNode(slot.name, slot.hlslType, slot.id, guid))
                                
    def getUniforms(self):
        uniforms = []
        for prop in self.properties.values():
            propType = prop.type
            if prop.overrideRefName != '':
                propName = prop.overrideRefName
            else:
                propName = prop.defaultRefName
            uniforms.append((propType, propName))
            
        for keyword in self.keywords.values():
            uniforms.append(('bool', keyword.overrideRefName[:-3]))
        
        sortedUniforms = []
        for u in uniforms:
            if u[0] == 'float4':
                sortedUniforms.append(u)
        for u in uniforms:
            if u[0] == 'float2':
                sortedUniforms.append(u)
        for u in uniforms:
            if u[0] == 'float':
                sortedUniforms.append(u)
        for u in uniforms:
            if u[0] == 'bool':
                sortedUniforms.append(u)
        for u in uniforms:
            if u[0] == 'TextureInfo':
                sortedUniforms.append(u)
        return sortedUniforms

    def getSubGraphs(self):
        subGraphIDs = set()
        for guid, node in self.nodes.items():
           if type(node) == SubGraphNode:
               subGraphIDs.add(node.guid)
        return subGraphIDs

    def parseGraph(self):
        
        q = queue.Queue()
        varIdx = 0
        
        for guid, node in self.nodes.items():
            isInputNode = True
            hasOutNodes = False
            for slot in node.slots:
                if slot.slotType == 0:
                    if (guid, slot.id) in self.edgeMapIn: # if the node has atleast one input connection its not a root node
                        isInputNode = False
                        break
                if slot.stage == 1: # if the node is a vertex shader output its not a root node
                    isInputNode = False 
                    break

            # TODO: this is a quick fix to not include vertex shader code
            #       need to split the nodes into the shader stages before parsing
            if isInputNode and node.type != 'UnityEditor.ShaderGraph.PositionNode':
                q.put(node)
                #print('found input node %s %s' % (node.name, node.type))

        codeLines = []
        while not q.empty():
            outNode = q.get()
            if len(outNode.getInputSlots()) == 0: # input nodes
                var = outNode.propName
                if var == '':
                    code, outName, outType = outNode.getCode(varIdx)
                    codeLines.extend(code)
                    varIdx += 1
                    outVars = outNode.getOutputVars(outName, outType)
                else:
                    type = outNode.slots[0].hlslType
                    outVars = [(var, type)]
            elif len(outNode.getOutputSlots()) == 0: # output nodes
                code, _, _ = outNode.getCode(varIdx)
                codeLines.extend(code)
                varIdx += 1
            else: # all other nodes
                code, outName, outType = outNode.getCode(varIdx)
                outVars = outNode.getOutputVars(outName, outType)
                if code[0] != '':
                    codeLines.extend(code)
                varIdx += 1
            for outSlot in outNode.getOutputSlots():
                if (outNode.id, outSlot) in self.edgeMapOut:
                    for nodeID, slotID in self.edgeMapOut[(outNode.id, outSlot)]:
                        nextNode = self.nodes[nodeID]
                        nextNode.visit(outVars[outSlot][0], outVars[outSlot][1], slotID)
                        if nextNode.finished:
                            q.put(nextNode)
        return codeLines

    def getCode(self):
        #print(self.name)
        funcName = self.name.replace('-', '')
        inputName = funcName + 'DataIn'
        outputName = funcName + 'DataOut'
        code = []
        code.append('struct ' + inputName + '\n')
        code.append('{\n')
        if len(self.properties) == 0:
            code.append('\tfloat dummy;\n')
        for prop in self.properties.values():
            code.append('\t' + prop.type + ' ' + prop.getName() + ';\n')
        code.append('};\n')
        code.append('\n')

        code.append('struct ' + outputName + '\n')
        code.append('{\n')
        
        for out in self.outputNodes:
            code.append('\t' + out.type + ' ' + out.name + ';\n')
        code.append('};\n')
        code.append('\n')
        
        code.append('void ' + funcName + '(Input i, inout ' + inputName + ' ' + self.inPrefix[:-1] + ', inout ' + outputName + ' ' + self.outPrefix[:-1] + ')\n')
        code.append('{\n')
        for l in self.parseGraph():
            code.append('\t' + l)
        code.append('}\n')
        return code

In [17]:
assetPath = 'C:/workspace/code/VikingVillage/Assets'
shaderList = []
subGraphs = {}
subshaders = []
for r,d,f in os.walk(assetPath):
    for fn in f:
        fullPath = r + '/' + fn
        fullPath = fullPath.replace('\\', '/')
        if fn.endswith('shadergraph'):
            shaderList.append((fn, fullPath))
        elif fn.endswith('shadersubgraph'):
            #subGraphList.append((fn, fullPath))
            with open(fullPath + '.meta', 'r') as stream:
                content = yaml.safe_load(stream)
                guid = content['guid']
                subGraphs[guid] = ((fn, fullPath))
                subshaders.append((fn, fullPath))

for shaderFn, shaderPath in shaderList:
    #shaderFn, shaderPath = shaderList[2]
    print('converting shader graph %s' % shaderFn)
    
    sg = ShaderGraph(shaderPath, '', 'o.')
    mainCode = sg.parseGraph()
    uniforms = sg.getUniforms()
    subGraphIDs = sg.getSubGraphs()
    subGraphCode = []
    for guid in subGraphIDs:
        if guid in subGraphs:
            subGraphPath = subGraphs[guid][1]
            subgraph = ShaderGraph(subGraphPath, 'inData.', 'outData.')
            subGraphCode.append(subgraph.getCode())
    
    shaderName = shaderFn.split('.')[0]
    
    with open('../HLSL/Materials/Unity/UnityMaterialTemplate.hlsl', 'r') as f:
        lines = f.readlines()
    
    outlines = []
    for line in lines:
        if line.find('@UnityMaterial') > -1:
            for u in uniforms:
                outlines.append('\t' + u[0] + ' ' + u[1] + ';\n')
        elif line.find('@UnitySurface') > -1:
            # for u in uniforms:
            #     outlines.append('#define ' + u[1] + ' material.' + u[1] + '\n')
            # outlines.append('\n')
            for code in subGraphCode:
                for l in code:
                    outlines.append(l)
            outlines.append('\n')
            outlines.append('void surf(Input i, inout SurfaceOutputStandardSpecular o)\n')
            outlines.append('{\n')
            for l in mainCode:
                outlines.append('\t' + l)
            outlines.append('}\n')
            outlines.append('\n')
            outlines.append('static Input unityInput;\n')
            outlines.append('static SurfaceOutputStandardSpecular unitySurface;\n')
        else:
            outlines.append(line)
    
    with open('../HLSL/Materials/Unity/Unity' + shaderName + '.hlsl', 'w') as f:
        f.writelines(outlines)

converting shader graph Standard.shadergraph
converting shader graph Standard_AlphaClip.shadergraph
converting shader graph TerrainSurfaceGraph.shadergraph
converting shader graph Lit_SSS_Cutout.shadergraph
error node type UnityEditor.ShaderGraph.CustomFunctionNode not implemented!
